# Epoch/Round Tuning Analysis

This notebook reads the grouped `epoch_round_tuning/tuning-*` runs, summarizes per-round validation MMLU, compares the selected epoch/round pairs across linear FLoRA, nonlinear FLoRA, and nonlinear FFA, and renders Plotly heatmaps for the epoch x round grid.

In [1]:
from pathlib import Path

from IPython.display import display

from utils.tuning_analysis import (
    build_extension_requests,
    build_llama_confirmation_requests,
    build_repeat_requests,
    compare_selected_to_paper,
    load_tuning_results,
    make_tuning_heatmap,
    make_tuning_round_curves,
    select_plateaus,
    summarize_tuning_results,
    write_manifest,
)

BASE_DIR = Path('.')
TUNING_RUN_ROOTS = [
    BASE_DIR / 'epoch_round_tuning',
    BASE_DIR / 'epoch_round_tuning_r10',
    BASE_DIR / 'epoch_round_tuning_dolly_dirichlet_r10',
    BASE_DIR / 'exp2-tinyllama-nonlinear-wiz',
    BASE_DIR / 'exp2-tinyllama-nonlinear-3round-heter-wiz',
    BASE_DIR / 'exp2-llama-nonlinear-1round-homo-wiz',
    BASE_DIR / 'exp2-llama-nonlinear-1round-heter-wiz',
]
OUTPUT_DIR = BASE_DIR / 'tuning_analysis_outputs'
FIGURE_DIR = OUTPUT_DIR / 'epoch_round_heatmaps'

OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

## Load And Select

In [2]:
tuning_scores = load_tuning_results(TUNING_RUN_ROOTS)
tuning_summary = summarize_tuning_results(tuning_scores)
tuning_plateaus, tuning_selected = select_plateaus(tuning_summary)

run_count = tuning_scores['Run dir'].nunique() if not tuning_scores.empty else 0
case_count = tuning_selected.shape[0] if not tuning_selected.empty else 0
print(f'Loaded {len(tuning_scores)} per-round scores from {run_count} runs under {TUNING_RUN_ROOTS}.')
print(f'Selected {case_count} method/dataset/model/setting configurations.')

display(tuning_selected.round(2))
display(tuning_plateaus.round(2))

Loaded 284 per-round scores from 36 runs under [PosixPath('epoch_round_tuning'), PosixPath('epoch_round_tuning_r10'), PosixPath('epoch_round_tuning_dolly_dirichlet_r10'), PosixPath('exp2-tinyllama-nonlinear-wiz'), PosixPath('exp2-tinyllama-nonlinear-3round-heter-wiz'), PosixPath('exp2-llama-nonlinear-1round-homo-wiz'), PosixPath('exp2-llama-nonlinear-1round-heter-wiz')].
Selected 12 method/dataset/model/setting configurations.


,Method key,Method,Dataset,Dataset label,Model key,Model,Setting key,Setting,Selected epochs,Selected round,Selected accuracy,Selected std,Selected seed count,Selected seeds,Selected compute cost,Best epochs,Best round,Best accuracy,Accuracy gap to best,Low-cost tie count
0,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,5,2,29.19,0.0,1,0,10,5,6,29.46,0.27,1
1,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,5,6,30.86,0.0,1,0,30,5,6,30.86,0.00,1
4,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,2,3,33.06,0.0,1,0,6,1,10,33.63,0.57,2
5,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,homo,Homo,1,4,31.50,0.0,1,0,4,5,1,32.19,0.69,1
2,flora,Linear FLoRA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,5,1,28.34,0.0,1,0,5,2,8,29.11,0.77,1
3,flora,Linear FLoRA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,1,5,28.63,0.0,1,0,5,5,4,29.61,0.98,1
6,flora,Linear FLoRA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,3,1,46.22,0.0,1,0,3,3,1,46.22,0.00,1
7,flora,Linear FLoRA,wiz,Wizard,tinyllama,TinyLlama,homo,Homo,1,1,45.10,0.0,1,0,1,1,1,45.10,0.00,1
8,nonlinear_flora,Nonlinear FLoRA,wiz,Wizard,llama,Llama-7B,heter,Heter,1,1,25.28,0.0,1,2,1,1,1,25.28,0.00,1
9,nonlinear_flora,Nonlinear FLoRA,wiz,Wizard,llama,Llama-7B,homo,Homo,1,1,27.25,0.0,1,2,1,1,1,27.25,0.00,1


,Method key,Method,Dataset,Dataset label,Model key,Model,Setting key,Setting,Local epochs,Plateau round,Plateau accuracy,Best round,Best accuracy,Max round observed,Declined after best
0,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,1,5,24.01,6,24.15,6,False
1,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,2,3,27.09,6,27.12,6,False
2,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,3,3,27.60,4,28.21,6,True
3,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,5,2,29.19,6,29.46,6,False
4,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,1,6,27.63,8,28.55,10,False
5,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,2,4,29.31,10,29.76,10,False
6,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,3,4,28.42,6,29.07,6,False
7,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,5,6,30.86,6,30.86,6,False
16,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,1,6,33.19,10,33.63,10,False
17,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,2,3,33.06,3,33.06,6,False


In [3]:
tuning_scores.to_csv(OUTPUT_DIR / 'tuning_scores.csv', index=False)
tuning_summary.to_csv(OUTPUT_DIR / 'tuning_summary.csv', index=False)
tuning_plateaus.to_csv(OUTPUT_DIR / 'tuning_plateaus.csv', index=False)
tuning_selected.to_csv(OUTPUT_DIR / 'tuning_selected.csv', index=False)
print(f'Wrote CSV summaries to {OUTPUT_DIR}.')

Wrote CSV summaries to tuning_analysis_outputs.


## Method Alignment

In [4]:
if tuning_selected.empty:
    print('No selected configurations yet.')
else:
    index_columns = ['Dataset label', 'Model', 'Setting']
    alignment = tuning_selected.pivot(
        index=index_columns,
        columns='Method',
        values=['Selected epochs', 'Selected round', 'Selected accuracy', 'Best epochs', 'Best round', 'Best accuracy'],
    )
    display(alignment.round(2))

    winners = (
        tuning_selected.sort_values('Selected accuracy', ascending=False)
        .groupby(index_columns, as_index=False)
        .first()[index_columns + ['Method', 'Selected epochs', 'Selected round', 'Selected accuracy', 'Best accuracy']]
    )
    display(winners.round(2))

Selected epochs                \
Method                                Linear FLoRA Nonlinear FFA   
Dataset label    Model     Setting                                 
Dolly stratified TinyLlama Heter               5.0           5.0   
                           Homo                1.0           5.0   
Wizard           Llama-7B  Heter               NaN           NaN   
                           Homo                NaN           NaN   
                 TinyLlama Heter               3.0           2.0   
                           Homo                1.0           1.0   

                                                   Selected round  \
Method                             Nonlinear FLoRA   Linear FLoRA   
Dataset label    Model     Setting                                  
Dolly stratified TinyLlama Heter               NaN            1.0   
                           Homo                NaN            5.0   
Wizard           Llama-7B  Heter               1.0            NaN   
                           Homo                1.0            NaN   
                 TinyLlama Heter               1.0            1.0   
                           Homo                1.0            1.0   

                                                                  \
Method                             Nonlinear FFA Nonlinear FLoRA   
Dataset label    Model     Setting                                 
Dolly stratified TinyLlama Heter             2.0             NaN   
                           Homo              6.0             NaN   
Wizard           Llama-7B  Heter             NaN             1.0   
                           Homo              NaN             1.0   
                 TinyLlama Heter             3.0             1.0   
                           Homo              4.0             1.0   

                                   Selected accuracy                \
Method                                  Linear FLoRA Nonlinear FFA   
Dataset label    Model     Setting                                   
Dolly stratified TinyLlama Heter               28.34         29.19   
                           Homo                28.63         30.86   
Wizard           Llama-7B  Heter                 NaN           NaN   
                           Homo                  NaN           NaN   
                 TinyLlama Heter               46.22         33.06   
                           Homo                45.10         31.50   

                                                    Best epochs                \
Method                             Nonlinear FLoRA Linear FLoRA Nonlinear FFA   
Dataset label    Model     Setting                                              
Dolly stratified TinyLlama Heter               NaN          2.0           5.0   
                           Homo                NaN          5.0           5.0   
Wizard           Llama-7B  Heter             25.28          NaN           NaN   
                           Homo              27.25          NaN           NaN   
                 TinyLlama Heter             35.94          3.0           1.0   
                           Homo              36.87          1.0           5.0   

                                                     Best round                \
Method                             Nonlinear FLoRA Linear FLoRA Nonlinear FFA   
Dataset label    Model     Setting                                              
Dolly stratified TinyLlama Heter               NaN          8.0           6.0   
                           Homo                NaN          4.0           6.0   
Wizard           Llama-7B  Heter               1.0          NaN           NaN   
                           Homo                1.0          NaN           NaN   
                 TinyLlama Heter               1.0          1.0          10.0   
                           Homo                1.0          1.0           1.0   

                                                   Best accuracy  \
Method                      

,Dataset label,Model,Setting,Method,Selected epochs,Selected round,Selected accuracy,Best accuracy
0,Dolly stratified,TinyLlama,Heter,Nonlinear FFA,5,2,29.19,29.46
1,Dolly stratified,TinyLlama,Homo,Nonlinear FFA,5,6,30.86,30.86
2,Wizard,Llama-7B,Heter,Nonlinear FLoRA,1,1,25.28,25.28
3,Wizard,Llama-7B,Homo,Nonlinear FLoRA,1,1,27.25,27.25
4,Wizard,TinyLlama,Heter,Linear FLoRA,3,1,46.22,46.22
5,Wizard,TinyLlama,Homo,Linear FLoRA,1,1,45.10,45.10


## Paper Baseline Comparison

The Wizard and Dolly paper baselines are taken from `flora_results_analysis.ipynb`. Paper results are plotted only at their reported communication round, not as horizontal lines across all rounds. Dolly tuning here uses the stratified split, so the Dolly paper point is contextual rather than an exact split-matched baseline.

In [5]:
paper_comparison = compare_selected_to_paper(tuning_selected)
if paper_comparison.empty:
    print('No paper baselines matched the selected tuning rows.')
else:
    paper_comparison.to_csv(OUTPUT_DIR / 'tuning_vs_paper.csv', index=False)
    display(paper_comparison.round(2))


,Method,Dataset,Model,Setting,Selected epochs,Selected round,Selected accuracy,Best epochs,Best round,Best accuracy,FLoRA paper,FLoRA paper round,Selected delta vs paper,Best delta vs paper
0,Nonlinear FFA,Dolly stratified,TinyLlama,Heter,5,2,29.19,5,6,29.46,18.45,3,10.74,11.01
1,Nonlinear FFA,Dolly stratified,TinyLlama,Homo,5,6,30.86,5,6,30.86,30.80,3,0.06,0.06
2,Nonlinear FFA,Wizard,TinyLlama,Heter,2,3,33.06,1,10,33.63,41.48,3,-8.42,-7.85
3,Nonlinear FFA,Wizard,TinyLlama,Homo,1,4,31.50,5,1,32.19,43.87,3,-12.37,-11.68
4,Linear FLoRA,Dolly stratified,TinyLlama,Heter,5,1,28.34,2,8,29.11,18.45,3,9.89,10.66
5,Linear FLoRA,Dolly stratified,TinyLlama,Homo,1,5,28.63,5,4,29.61,30.80,3,-2.17,-1.19
6,Linear FLoRA,Wizard,TinyLlama,Heter,3,1,46.22,3,1,46.22,41.48,3,4.74,4.74
7,Linear FLoRA,Wizard,TinyLlama,Homo,1,1,45.10,1,1,45.10,43.87,3,1.23,1.23
8,Nonlinear FLoRA,Wizard,Llama-7B,Heter,1,1,25.28,1,1,25.28,27.91,1,-2.63,-2.63
9,Nonlinear FLoRA,Wizard,Llama-7B,Homo,1,1,27.25,1,1,27.25,34.26,1,-7.01,-7.01


## Round Curves

In [6]:
if tuning_summary.empty:
    print('No tuning summary to plot.')
else:
    for dataset in sorted(tuning_summary['Dataset'].unique()):
        for model in sorted(tuning_summary['Model key'].unique()):
            fig = make_tuning_round_curves(
                tuning_summary,
                dataset=dataset,
                model=model,
                include_paper_baseline=True,
            )
            if fig is not None:
                fig.show()

/home/trieu.vy.tran/FederatedLLM/utils/tuning_analysis.py:681: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  plot_data = pd.concat([plot_data, pd.DataFrame(baseline_rows)], ignore_index=True)


/home/trieu.vy.tran/FederatedLLM/utils/tuning_analysis.py:681: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  plot_data = pd.concat([plot_data, pd.DataFrame(baseline_rows)], ignore_index=True)


/home/trieu.vy.tran/FederatedLLM/utils/tuning_analysis.py:681: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  plot_data = pd.concat([plot_data, pd.DataFrame(baseline_rows)], ignore_index=True)


## Epoch x Round Heatmaps

In [7]:
# if tuning_summary.empty:
#     print('No tuning summary to plot.')
# else:
#     heatmap_cases = (
#         tuning_summary[['Method key', 'Dataset', 'Model key', 'Setting key']]
#         .drop_duplicates()
#         .sort_values(['Dataset', 'Model key', 'Setting key', 'Method key'])
#     )
#     for _, row in heatmap_cases.iterrows():
#         fig = make_tuning_heatmap(
#             tuning_summary,
#             method=row['Method key'],
#             dataset=row['Dataset'],
#             model=row['Model key'],
#             setting=row['Setting key'],
#         )
#         if fig is None:
#             continue
#         html_name = f"heatmap_{row['Method key']}_{row['Dataset']}_{row['Model key']}_{row['Setting key']}.html"
#         fig.write_html(FIGURE_DIR / html_name)
#         fig.show()
#     print(f'Wrote heatmap HTML files to {FIGURE_DIR}.')
